# Bài 23: Xóa tài liệu khỏi kho
## Cho user quản lý kho — thêm được thì phải xóa được

**Mục tiêu:**
- Viết `delete_document()` — xóa mọi chunk của một tài liệu
- Thêm nút 🗑️ cạnh mỗi file trong mục '📚 Kho tài liệu'
- Hiểu `st.columns` để đặt nút cạnh nhãn

**Vì sao cần:** kho do user tự xây thì phải cho họ dọn — upload nhầm, tài liệu cũ, hay muốn thay báo cáo mới. Thiếu nút xóa là thiếu nửa vòng đời quản lý dữ liệu.

---
## Phần 1: Lý thuyết

### Xóa trong ChromaDB — bạn đã dùng rồi

Hàm ingest idempotent (bài 19 nâng cấp) đã dùng đúng cơ chế này để xóa bản cũ trước khi thêm:

```python
collection.delete(where={"$and": [{"symbol": symbol}, {"source": source_name}]})
```

`where` filter y hệt lúc query. Xóa 1 tài liệu = xóa mọi chunk khớp `(symbol, source)`.

| Muốn xóa | where |
|----------|-------|
| 1 file cụ thể | `{"$and": [{"symbol": sym}, {"source": src}]}` |
| Cả 1 công ty | `{"symbol": sym}` |
| Sạch toàn bộ | `client.delete_collection(name)` |

---

### `st.columns` — đặt nút cạnh nhãn

Để nút 🗑️ nằm bên phải tên file, chia hàng thành 2 cột:

```python
col1, col2 = st.columns([5, 1])   # tỉ lệ rộng 5:1
col1.caption("• NVDA-10Q.pdf")
if col2.button("🗑️", key="..."):
    # xóa
```

---

### Mỗi nút cần `key` riêng + `st.rerun()` sau khi xóa

- **`key` riêng**: nhiều nút cùng label "🗑️" → phải phân biệt bằng key, ví dụ `key=f"del::{sym}::{file}"`. Không có key riêng → Streamlit báo lỗi 'duplicate button'.
- **`st.rerun()`**: sau khi xóa, chạy lại để danh sách kho cập nhật ngay (file vừa xóa biến mất).

---
## Phần 2: Ví dụ — add rồi delete (in-memory)

Dùng ChromaDB in-memory để thấy `delete(where=...)` hoạt động, không đụng kho thật.

In [1]:
import chromadb

client = chromadb.Client()  # in-memory
col = client.create_collection("demo_delete")

# Thêm 3 chunk: 2 của NVDA (2 file khác nhau) + 1 của AMD
col.add(
    documents=["nvda 10q chunk", "nvda slide chunk", "amd 10q chunk"],
    metadatas=[
        {"symbol": "NVDA", "source": "NVDA-10Q.pdf"},
        {"symbol": "NVDA", "source": "NVDA-slides.pdf"},
        {"symbol": "AMD",  "source": "AMD-10Q.pdf"},
    ],
    ids=["n1", "n2", "a1"],
)
print("Trước khi xóa:", col.count(), "chunks")

# Xóa ĐÚNG 1 file: NVDA-10Q.pdf
col.delete(where={"$and": [{"symbol": "NVDA"}, {"source": "NVDA-10Q.pdf"}]})
print("Sau khi xóa NVDA-10Q.pdf:", col.count(), "chunks")

# Kiểm tra còn lại
for m in col.get(include=["metadatas"])["metadatas"]:
    print("  còn:", m["symbol"], "|", m["source"])

Trước khi xóa: 3 chunks
Sau khi xóa NVDA-10Q.pdf: 2 chunks
  còn: NVDA | NVDA-slides.pdf
  còn: AMD | AMD-10Q.pdf


Quan sát: chỉ `NVDA-10Q.pdf` bị xóa, `NVDA-slides.pdf` và `AMD-10Q.pdf` còn nguyên. Đó là độ chính xác của filter `$and`.

---
## Phần 3: Bài tập

### Bước 1: Thêm `delete_document()` vào `app/services/rag.py`

Đặt cạnh `get_kb_status()` (đều là hàm quản lý kho). Điền `TODO`:

```python
def delete_document(symbol: str, source: str) -> None:
    """Xóa toàn bộ chunk của một tài liệu (theo symbol + source) khỏi kho."""
    _, collection = get_resources()
    # TODO: gọi collection.delete với where filter khớp CẢ symbol lẫn source
    #   (giống hàm ingest idempotent — dùng $and)
```

---

### Bước 2: Thêm nút xóa vào mục '📚 Kho tài liệu' — `app/main.py`

Thêm import: `from services.rag import get_kb_status, delete_document`

Sửa vòng lặp hiển thị kho (chỗ `for sym, files in kb.items()`), thêm nút 🗑️ cạnh mỗi file:

```python
    else:
        for sym, files in kb.items():
            st.markdown(f"**{sym}** — {len(files)} tài liệu")
            for f in files:
                col1, col2 = st.columns([5, 1])
                col1.caption(f"• {f}")
                # TODO: nút xóa với key riêng theo (sym, f)
                #   if col2.button("🗑️", key=f"del::{sym}::{f}"):
                #       delete_document(sym, f)
                #       st.rerun()
```

---

### Bước 3: Test

`streamlit run app/main.py`:
1. Đảm bảo kho có vài tài liệu (upload nếu cần)
2. Ở mục 📚 Kho tài liệu, bấm 🗑️ cạnh 1 file
3. File đó biến mất khỏi danh sách ngay (nhờ `st.rerun`)
4. Hỏi lại về công ty vừa xóa → không còn dữ liệu báo cáo của file đó

**Checklist:**
- [ ] Mỗi file có nút 🗑️ riêng, bấm đúng file nào xóa file đó
- [ ] Xóa xong danh sách cập nhật ngay
- [ ] Xóa file cuối của 1 công ty → công ty đó biến mất khỏi kho

**Suy ngẫm:** nếu muốn thêm nút "Xóa cả công ty" (mọi file của 1 symbol) thì `where` sẽ viết thế nào? (Gợi ý: bảng ở Phần 1.)